# Meridian — Return Head Training (Colab)

Trains the `MeridianR` return-prediction head on top of the pre-trained backbone.
Checkpoints save to Google Drive every 5000 steps. Re-run all cells to resume after a session drop.

**Joint FDA synthetic shards are generated locally via `make fda-joint`** (100-asset `FDA_SYNTH_*.f32bin` windows) and pushed to `meridian-data/fda_joint_tars/`; this notebook downloads and unpacks them, then mixes them in at `--warmup-fda-mix-prob 0.5` during the asset-anonymous correlation warm-up and `--fda-mix-prob 0.25` during head training (loss weight `--fda-loss-weight 0.6`).

The cross-asset warm-up is 15,000 steps (~13 h on A100-80GB at the measured ~0.3 it/s), checkpointed every 750 steps and resumable across sessions. Watch the `corr_frob` trend per source; if it is still clearly falling at 15k, resume with a higher `--cross-asset-warmup-steps`.

**Run this before `colab_portfolio_sharpe.ipynb`.**

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

# Rerun/iteration controls — code always re-installs fresh (see cell 3);
# these force-refresh the cached artifacts too when you've republished them.
FORCE_SYNTH_REFRESH = False   # True → re-download joint FDA synth tars
FORCE_CKPT_REFRESH  = False   # True → re-fetch pretrain backbone from HF over the Drive copy

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/pretrain',  exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints/portfolio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',            exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file (always fresh) ──────────────────
# Code is re-installed on every run so notebook reruns pick up the latest push
# to the HF repo — critical when iterating on portfolio_train.py locally.
# The copy is tiny (<1 MB); data and checkpoints stay cached.
import os, shutil

if os.path.exists('/content/meridian'):
    shutil.rmtree('/content/meridian')
    print('Removed stale code copy.')

local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
print('Code installed fresh.')

# portfolio_symbols.txt lives at repo root, not inside code/, so fetch it separately.
symbols_path = '/content/meridian/portfolio_symbols.txt'
path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='portfolio_symbols.txt')
shutil.copy(path, symbols_path)
print(f'portfolio_symbols.txt → {symbols_path}')

In [ ]:
# ── 4. Download data (cached to Drive across sessions) ────────────────────────
# Real shards live in meridian-data (train/validation/test); joint FDA synth
# tars are generated locally via `make fda-joint` and pushed to
# meridian-data/fda_joint_tars/ — the next cell pulls and unpacks them. On cold
# start with an empty Drive we snapshot_download the real shards here.
#
# Drive FUSE is flaky on dirs with 50k+ files (sporadic EIO on listdir). All
# Drive directory operations here go through a retry wrapper.
import os, shutil, time

def _drive_retry(fn, what, attempts=6, base_sleep=3):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except OSError as e:
            last = e
            print(f'[{what}] Drive EIO (attempt {i + 1}/{attempts}): {e}; sleeping {(i + 1) * base_sleep}s')
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(
        f'{what} failed after {attempts} retries. Drive FUSE is likely stuck. '
        f'Remount with: from google.colab import drive; drive.flush_and_unmount(); '
        f'drive.mount("/content/drive", force_remount=True); then re-run.'
    ) from last

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'

def _has_data():
    if not os.path.exists(train_dir):
        return False
    # any() over listdir — wrapped so the 50k-file listing retries through EIO.
    entries = _drive_retry(lambda: os.listdir(train_dir), 'list train_dir')
    return any(f.endswith('.f32bin') for f in entries)

has_data = _has_data()

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir,
                      allow_patterns=['train/**', 'validation/**', 'test/**'])
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────────────────
# Parallel copy: shutil.copytree is single-threaded, which means each of the
# ~50k Drive-side files waits on its own round-trip. Drive FUSE handles
# concurrent reads well, so a thread pool gets 8–16× speedup. Each file copy
# is retried on EIO because FUSE occasionally chokes on hot directories.
import os, shutil, time, concurrent.futures

local_data = '/content/data/mmaps'
drive_data = f'{DRIVE_DIR}/data/mmaps'

def _copy_one(src, dst, attempts=5, base_sleep=1):
    last = None
    for i in range(attempts):
        try:
            if os.path.exists(dst):
                return  # idempotent — don't re-copy on rerun
            shutil.copy2(src, dst)
            return
        except OSError as e:
            last = e
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(f'copy failed {src} → {dst}: {last}') from last

def _parallel_copy_dir(src_dir, dst_dir, workers=32):
    os.makedirs(dst_dir, exist_ok=True)
    entries = sorted(os.listdir(src_dir))
    jobs = [(os.path.join(src_dir, f), os.path.join(dst_dir, f)) for f in entries]
    t0 = time.time()
    done = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_copy_one, s, d) for s, d in jobs]
        for fut in concurrent.futures.as_completed(futures):
            fut.result()
            done += 1
            if done % 2000 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                print(f'    {done}/{len(jobs)} ({rate:.0f} files/s)')
    print(f'  {src_dir} → {dst_dir}: {len(jobs)} files in {time.time() - t0:.1f}s')

if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (parallel, one-time per session)...')
    # Copy each split separately — the train/ dir is the slow one (50k files),
    # validation and test are tiny.
    for split in ('train', 'validation', 'test'):
        src_split = os.path.join(drive_data, split)
        if os.path.isdir(src_split):
            _parallel_copy_dir(src_split, os.path.join(local_data, split))
    print('Done.')
else:
    print('Local data already present.')

# Report shard breakdown — portfolio_train.py auto-detects FDA_SYNTH_* shards
# and mixes them into head training. If count is 0, generate them via
# `make fda-joint` first.
train_files = [f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin')]
real_shards  = sorted(f for f in train_files if not f.startswith('FDA_SYNTH_'))
print(f'Real train shards: {len(real_shards)} (joint FDA synth is fetched into --synth-dir by the next cell)')


In [ ]:
# ── Download + unpack JOINT FDA synth tars into a dedicated --synth-dir ────────
# Joint FDA_SYNTH_*.f32bin windows (100-asset (n_assets,T,F), generated locally
# via `make fda-joint`) are packed into ~100 MiB tars under fda_joint_tars/ in
# the data repo. They go in their OWN dir (NOT the real-shard data-dir): the
# portfolio trainer reads them via --synth-dir, because per-asset and joint synth
# share the FDA_SYNTH_ prefix but have different shapes.
import os, tarfile, shutil
from huggingface_hub import list_repo_files, hf_hub_download

synth_dir   = '/content/data/mmaps_synth_joint'
synth_train = f'{synth_dir}/train'
os.makedirs(synth_train, exist_ok=True)

existing_synth = [f for f in os.listdir(synth_train) if f.startswith('FDA_SYNTH_')]
if existing_synth and FORCE_SYNTH_REFRESH:
    for f in existing_synth:
        os.remove(os.path.join(synth_train, f))
    print(f'FORCE_SYNTH_REFRESH: cleared {len(existing_synth)} cached synth shards.')
    existing_synth = []

if existing_synth:
    print(f'Joint FDA synth shards already present: {len(existing_synth)} — skipping download.')
else:
    try:
        tar_files = [f for f in list_repo_files(HF_DATA_REPO, repo_type='dataset')
                     if f.startswith('fda_joint_tars/') and f.endswith('.tar')]
    except Exception as e:
        tar_files = []
        print(f'Could not list {HF_DATA_REPO}: {e}')
    if not tar_files:
        print('No joint FDA tars found — training will run real-data-only. '
              'Generate via `make fda-joint` locally and upload to '
              'meridian-data/fda_joint_tars/.')
    else:
        print(f'Downloading and unpacking {len(tar_files)} joint FDA tar(s) from {HF_DATA_REPO}...')
        extracted = 0
        for tar_name in sorted(tar_files):
            local_tar = hf_hub_download(repo_id=HF_DATA_REPO, filename=tar_name, repo_type='dataset')
            with tarfile.open(local_tar, 'r') as tf:
                for member in tf:
                    if not member.isfile() or not member.name.endswith('.f32bin'):
                        continue
                    dst = os.path.join(synth_train, os.path.basename(member.name))
                    if os.path.exists(dst):
                        continue
                    with tf.extractfile(member) as sf, open(dst, 'wb') as df:
                        shutil.copyfileobj(sf, df)
                    extracted += 1
            os.remove(local_tar)
        print(f'Extracted {extracted} joint FDA synth shards.')

synth_shards = [f for f in os.listdir(synth_train) if f.startswith('FDA_SYNTH_')]
print(f'Joint FDA synth shards in {synth_train}: {len(synth_shards)}')

In [ ]:
# ── 6. Locate pretrain checkpoint (prefers best-val, then latest step, then HF) ─
# Portfolio training *consumes* the backbone as a fixed initialisation, so we
# want the highest-quality artifact: backbone_pretrain.pt is saved only on
# val-loss improvement (best), whereas backbone_step*.pt are unconditional
# periodic saves that may be staler/worse. Prefer best-val; fall back to the
# latest step checkpoint only if no best-val exists locally or on HF.
import os, re, glob, shutil

PRETRAIN_DIR   = f'{DRIVE_DIR}/checkpoints/pretrain'
pretrain_best  = f'{PRETRAIN_DIR}/backbone_pretrain.pt'

def _latest_step_ckpt(ckpt_dir):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, 'backbone_step*.pt')):
        m = re.search(r'backbone_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

if FORCE_CKPT_REFRESH and os.path.exists(pretrain_best):
    os.remove(pretrain_best)
    print('FORCE_CKPT_REFRESH: removed Drive backbone copy; re-fetching from HF.')

# Fetch best-val from HF if Drive doesn't already have it.
if not os.path.exists(pretrain_best):
    try:
        path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='pretrain/backbone_pretrain.pt')
        shutil.copy(path, pretrain_best)
        print(f'Pretrain best-val checkpoint downloaded from HF → {pretrain_best}')
    except Exception as e:
        print(f'best-val not available on HF ({e}); will try a local step checkpoint.')

if os.path.exists(pretrain_best):
    PRETRAIN_CKPT = pretrain_best
    print(f'Will init backbone from best-val checkpoint: {PRETRAIN_CKPT}')
else:
    latest_step = _latest_step_ckpt(PRETRAIN_DIR)
    if latest_step is not None:
        PRETRAIN_CKPT = latest_step
        print(f'No best-val found; falling back to latest step checkpoint: {PRETRAIN_CKPT}')
    else:
        raise RuntimeError('No pretrain checkpoint found.')

In [ ]:
# ── 7. Auto-detect GPU ────────────────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# TF32 matmul — free speedup on Ampere+ with negligible numerical impact.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# QUALITY-FIRST config (project priority: quality over efficiency). Cheap quality
# wins are global; the expensive one — full-sequence BPTT — is gated to 80GB.
#   • stride=1, regime_pool=128, eval_every=100 — denser data, tighter regime
#     balance, finer best-DSR checkpoint selection (all near-free on memory).
#   • bptt_chunk=0 (FULL 64-step BPTT) is the top temporal-quality lever but is
#     batch-bound: over 100 assets the full-seq graph is ~one backbone forward of
#     batch×100×64. Gradient checkpointing only rematerialises across LAYER DEPTH,
#     not the time axis (one chunk = full seq), so the per-backward batch is the
#     binding constraint. On 80GB it fits only at warmup_batch=4 / pta_micro=4.
#   • pta_micro_batch_size is grad-accumulated → BIT-EXACT identical to a big
#     batch (pure memory knob). warmup_batch only adds gradient noise to a
#     prior-shaping stage. Both are safe to shrink for memory.
#   • PTA §11.4b expands B up to 10× with --pta-deterministic (fixed B_eff,
#     compile-friendly, strictly stronger regularisation).
PTA_DETERMINISTIC = True
DISABLE_REGIME_BALANCE = False
USE_COMPILE = True
REGIME_POOL_SIZE = 128         # quality: tighter regime balance (low memory cost)
STRIDE = 1                     # quality: 4× denser training sequences
EVAL_EVERY = 100               # quality: finer-grained best-DSR checkpoint selection

if vram_gb >= 75:    # A100/H100 80GB — FULL-sequence BPTT (max temporal quality)
    BPTT_CHUNK    = 0
    USE_GRADIENT_CHECKPOINTING = True
    BATCH_SIZE    = 6
    WARMUP_BATCH_SIZE = 4      # full-seq graph over 100 assets caps this at ~4
    PTA_MICRO_BS  = 4          # per-backward group; >4 OOMs with full BPTT
elif vram_gb >= 35:  # A100 40GB — long (48-step) truncated BPTT
    BPTT_CHUNK    = 48
    USE_GRADIENT_CHECKPOINTING = False
    BATCH_SIZE    = 4
    WARMUP_BATCH_SIZE = 8
    PTA_MICRO_BS  = 4
elif vram_gb >= 22:  # L4 24GB
    BPTT_CHUNK    = 8
    USE_GRADIENT_CHECKPOINTING = False
    BATCH_SIZE    = 4
    WARMUP_BATCH_SIZE = 8
    PTA_MICRO_BS  = 2
else:                # T4/V100 16GB
    BPTT_CHUNK    = 2
    USE_GRADIENT_CHECKPOINTING = False
    BATCH_SIZE    = 2
    WARMUP_BATCH_SIZE = 8
    PTA_MICRO_BS  = 1
    REGIME_POOL_SIZE = 64
    DISABLE_REGIME_BALANCE = True
    print('WARNING: 16GB tier — bptt_chunk=2, B=2, regime-balance off (quality/spec deviation).')

WORKERS = min(8, os.cpu_count() or 1)
print(
    f'batch={BATCH_SIZE}  warmup_batch={WARMUP_BATCH_SIZE}  bptt_chunk={BPTT_CHUNK}  '
    f'grad_ckpt={USE_GRADIENT_CHECKPOINTING}  stride={STRIDE}  eval_every={EVAL_EVERY}  '
    f'pta_micro_bs={PTA_MICRO_BS}  pta_deterministic={PTA_DETERMINISTIC}  '
    f'regime_pool={REGIME_POOL_SIZE}  compile={USE_COMPILE}  '
    f'regime_balance={not DISABLE_REGIME_BALANCE}  workers={WORKERS}'
)

In [ ]:
# ── 8. Train return head (auto-resumes from Drive) ────────────────────────────
import os, sys, subprocess

# Set DEBUG=True to print per-step [mem-diag] traces (GPU memory peaks per
# micro-group, first 3 steps only). Useful when tuning VRAM headroom.
DEBUG = False

import re, glob

CKPT_PORT      = f'{DRIVE_DIR}/checkpoints/portfolio'
return_best    = f'{CKPT_PORT}/return_head_best.pt'
warmup_ckpt    = f'{CKPT_PORT}/return_cross_asset_warmup.pt'

# Resume priority (mirrors colab_pretrain.ipynb): prefer the most recent step
# checkpoint so training continues from the last periodic save rather than the
# (possibly older) best-val checkpoint.
#   1. return_head_stepN.pt (highest N) — most recent periodic save
#   2. return_head_best.pt — best val monitor
#   3. return_cross_asset_warmup.pt — warmup done or partial, resumes in place
#   4. pretrain backbone — cold start (will run warmup first)
def _latest_step_ckpt(ckpt_dir, head):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, f'{head}_head_step*.pt')):
        m = re.search(rf'{head}_head_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

latest_step = _latest_step_ckpt(CKPT_PORT, 'return')
if latest_step is not None:
    resume_from = latest_step
    print(f'Resuming from latest step checkpoint: {resume_from}')
elif os.path.exists(return_best):
    resume_from = return_best
    print(f'Resuming from best-val checkpoint: {resume_from}')
elif os.path.exists(warmup_ckpt):
    resume_from = warmup_ckpt
    print(f'Resuming from cross-asset warmup: {resume_from}')
else:
    resume_from = PRETRAIN_CKPT
    print(f'Cold start from pretrain backbone: {resume_from}')

cmd = [
    sys.executable, '-u',
    '-m', 'train.portfolio_train',
    '--head',                          'return',
    '--data-dir',                      '/content/data/mmaps',
    '--synth-dir',                     '/content/data/mmaps_synth_joint',
    '--symbols-file',                  '/content/meridian/portfolio_symbols.txt',
    '--steps',                         '50000',
    '--cross-asset-warmup-steps',      '15000',
    '--fda-mix-prob',                  '0.25',
    '--warmup-fda-mix-prob',           '0.5',
    '--batch-size',                    str(BATCH_SIZE),
    '--cross-asset-warmup-batch-size', str(WARMUP_BATCH_SIZE),
    '--seq-len',                       '64',
    '--lr',                            '3e-5',
    '--log-every',                     '10',
    '--eval-every',                    str(EVAL_EVERY),
    '--save-every',                    '250',
    '--bptt-chunk',                    str(BPTT_CHUNK),
    '--stride',                        str(STRIDE),
    '--pta-micro-batch-size',          str(PTA_MICRO_BS),
    '--regime-pool-size',              str(REGIME_POOL_SIZE),
    '--checkpoint-dir',                CKPT_PORT,
    '--resume-backbone',               resume_from,
    '--workers',                       str(WORKERS),
]
if USE_GRADIENT_CHECKPOINTING:
    cmd.append('--gradient-checkpointing')
if PTA_DETERMINISTIC:
    cmd.append('--pta-deterministic')
if USE_COMPILE:
    cmd.append('--compile')
if DISABLE_REGIME_BALANCE:
    cmd.append('--disable-regime-balance')
if DEBUG:
    cmd.append('--debug')

env = os.environ.copy()
env['PYTHONPATH']         = '/content/meridian'
env['PYTHONUNBUFFERED']   = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed with return code {process.returncode}')
print('Training completed.')